В рамках этого практического задания вам будет дан код с кратким описанием. Вам необходимо описать какие из SOLID-принципов в нём нарушаются и отрефакторить его, приведя его в соответствие SOLID-принципам.

P.S. Решение всех задач не единственное. Необходимость любого рефакторинга на соответствие SOLID-принципам безусловно зависит от контекста и не является 100% необходимостью.

0. **Управление товарами на складе** (0б, с разбором)

Система для управления товарами на складе. Класс `WarehouseManager` выполняет обработку всех операций: добавление товара, удаление товара, генерация отчетов и синхронизация данных с удаленным сервером.

Исходный код:

In [1]:
import requests

class WarehouseManager:
    def __init__(self):
        self.items = {}

    def add_item(self, item_id, quantity):
        if item_id in self.items:
            self.items[item_id] += quantity
        else:
            self.items[item_id] = quantity
        print(f"Added {quantity} of item {item_id} to warehouse.")

    def remove_item(self, item_id, quantity):
        if item_id not in self.items or self.items[item_id] < quantity:
            raise ValueError("Insufficient stock")
        self.items[item_id] -= quantity
        print(f"Removed {quantity} of item {item_id} from warehouse.")

    def generate_report(self):
        report = "\n".join(f"{item_id}: {quantity}" for item_id, quantity in self.items.items())
        print("Warehouse Report:\n", report)
        return report

    def sync_with_server(self):
        response = requests.post("https://example.com/sync", json=self.items)
        if response.status_code == 200:
            print("Sync successful")
        else:
            print("Sync failed")

manager = WarehouseManager()
manager.add_item("item1", 10)
manager.remove_item("item1", 5)
manager.generate_report()
manager.sync_with_server()


Added 10 of item item1 to warehouse.
Removed 5 of item item1 from warehouse.
Warehouse Report:
 item1: 5
Sync failed


Проблемы:

1. **Нарушение SRP**:
   - `WarehouseManager` занимается управлением запасами, генерацией отчетов и синхронизацией с сервером.
   - Каждая из этих задач требует изменений в одном классе.

2. **Нарушение OCP**:
   - Если нужно добавить новую функциональность, например, расчет стоимости товаров, потребуется изменить `WarehouseManager`.

3. **Нарушение DIP**:
   - Класс напрямую зависит от библиотеки `requests` для синхронизации.

Решение:

In [2]:
class Inventory:
    def __init__(self):
        self.items = {}

    def add_item(self, item_id, quantity):
        if item_id in self.items:
            self.items[item_id] += quantity
        else:
            self.items[item_id] = quantity

    def remove_item(self, item_id, quantity):
        if item_id not in self.items or self.items[item_id] < quantity:
            raise ValueError("Insufficient stock")
        self.items[item_id] -= quantity

    def get_items(self):
        return self.items

class ReportGenerator:
    @staticmethod
    def generate_report(items):
        report = "\n".join(f"{item_id}: {quantity}" for item_id, quantity in items.items())
        print("Warehouse Report:\n", report)
        return report

class ServerSync:
    def __init__(self, server_url, client):
        self.server_url = server_url
        self.client = client

    def sync(self, data):
        response = self.client.post(self.server_url, json=data)
        if response.status_code == 200:
            print("Sync successful")
        else:
            print("Sync failed")

import requests

# Используем класс Inventory
inventory = Inventory()
inventory.add_item("item1", 10)
inventory.add_item("item2", 5)
inventory.remove_item("item1", 3)

# Генерация отчета
report = ReportGenerator.generate_report(inventory.get_items())

# Синхронизация с сервером
server_sync = ServerSync("https://example.com/sync", requests)
server_sync.sync(inventory.get_items())

Warehouse Report:
 item1: 7
item2: 5
Sync failed


1. **Система управления подписками (2б)**

В системе управления подписками есть разные планы подписок: базовый, премиум и бизнес. Каждому плану соответствует своя логика расчета стоимости с учетом скидок и налогов.

Исходный код:

In [ ]:
class SubscriptionManager:
    def __init__(self):
        self.plans = {}

    def add_plan(self, name, cost):
        self.plans[name] = {"cost": cost}

    def calculate_price(self, plan_name, tax_rate, discount=0):
        if plan_name not in self.plans:
            raise ValueError("Plan not found")
        base_price = self.plans[plan_name]["cost"]
        if plan_name == "premium":
            discount += 0.1  # Дополнительная скидка для премиум-плана
        elif plan_name == "business":
            discount += 0.2  # Дополнительная скидка для бизнес-плана
        final_price = base_price * (1 - discount) * (1 + tax_rate)
        return final_price

manager = SubscriptionManager()
manager.add_plan("basic", 10)
manager.add_plan("premium", 20)
manager.add_plan("business", 30)

print(manager.calculate_price("basic", 0.2))
print(manager.calculate_price("premium", 0.2))
print(manager.calculate_price("business", 0.2))

12.0
21.599999999999998
28.799999999999997


Проблемы:

1. **Нарушение SRP**:
   - Класс `SubscriptionManager` выполняет несколько обязанностей: управление списком планов подписок, расчёт стоимости, логика расчёта налогов и скидок смешана в одном методе

2. **Нарушение OCP**:
   - Метод `calculate_price` использует условные операторы (if/elif) для проверки типа плана подписки. Если понадобится добавить новый тип подписки с новой логикой расчёта скидки, придётся изменять существующий метод `calculate_price`

3. **Нарушение DIP**:
   - Класс `SubscriptionManager` завязан на конкретные реализации логики расчёта стоимости. Нет абстракций для расчёта скидок или налогов

Решение:

In [4]:
from abc import ABC, abstractmethod

class DiscountStrategy(ABC):
    @abstractmethod
    def get_discount(self, base_discount):
        pass


class BasicDiscount(DiscountStrategy):
    def get_discount(self, base_discount):
        return base_discount


class PremiumDiscount(DiscountStrategy):
    def get_discount(self, base_discount):
        return base_discount + 0.1


class BusinessDiscount(DiscountStrategy):
    def get_discount(self, base_discount):
        return base_discount + 0.2


class SubscriptionPlan:
    def __init__(self, name, cost, discount_strategy):
        self.name = name
        self.cost = cost
        self.discount_strategy = discount_strategy

    def calculate_price(self, tax_rate, discount = 0):
        total_discount = self.discount_strategy.get_discount(discount)
        return self.cost * (1 - total_discount) * (1 + tax_rate)


class SubscriptionManager:
    def __init__(self):
        self.plans = {}

    def add_plan(self, plan: SubscriptionPlan):
        self.plans[plan.name] = plan

    def get_plan(self, plan_name):
        if plan_name not in self.plans:
            raise ValueError("Plan not found")
        return self.plans[plan_name]

manager = SubscriptionManager()
manager.add_plan(SubscriptionPlan("basic", 10, BasicDiscount()))
manager.add_plan(SubscriptionPlan("premium", 20, PremiumDiscount()))
manager.add_plan(SubscriptionPlan("business", 30, BusinessDiscount()))

print(manager.get_plan("basic").calculate_price(0.2))
print(manager.get_plan("premium").calculate_price(0.2))
print(manager.get_plan("business").calculate_price(0.2))

12.0
21.599999999999998
28.799999999999997


2. **Система бронирования билетов (2б)**

В системе бронирования билетов есть разные типы билетов: эконом (невозвратные) и бизнес (возвратные). Каждый тип билета имеет свою стоимость и условия возврата.

Исходный код:

In [ ]:
class TicketBookingSystem:
    def __init__(self):
        self.tickets = []

    def book_ticket(self, ticket_type, cost, refundable):
        self.tickets.append({"type": ticket_type, "cost": cost, "refundable": refundable})
        print(f"Booked {ticket_type} ticket for ${cost}. Refundable: {refundable}")

    def process_refund(self, ticket_id):
        ticket = self.tickets[ticket_id]
        if not ticket["refundable"]:
            raise ValueError("This ticket is non-refundable")
        print(f"Refund processed for {ticket['type']} ticket costing ${ticket['cost']}")

    def charge_payment(self, amount):
        print(f"Charging ${amount} through payment gateway...")

system = TicketBookingSystem()
system.book_ticket("economy", 100, False)
system.book_ticket("business", 300, True)
system.process_refund(1)
system.charge_payment(400)

Booked economy ticket for $100. Refundable: False
Booked business ticket for $300. Refundable: True
Refund processed for business ticket costing $300
Charging $400 through payment gateway...


Проблемы:

1. **Нарушение SRP**:
   - Класс `TicketBookingSystem` выполняет много обязанностей, который можно разделить

2. **Нарушение OCP**:
   - Для добавления новых правил возврата нужно изменять существующий метод `process_refund`

3. **Нарушение DIP**:
   - Класс `TicketBookingSystem` напрямую работает со словарями вместо абстракций билетов

Решение:

In [7]:
from abc import ABC, abstractmethod

class Ticket(ABC):
    @abstractmethod
    def get_cost(self):
        pass

    @abstractmethod
    def is_refundable(self):
        pass


class EconomyTicket(Ticket):
    def __init__(self, cost):
        self.cost = cost

    def get_cost(self):
        return self.cost

    def is_refundable(self):
        return False


class BusinessTicket(Ticket):
    def __init__(self, cost):
        self.cost = cost

    def get_cost(self):
        return self.cost

    def is_refundable(self):
        return True


class PaymentProcessor:
    def charge(self, amount):
        print(f"Charging ${amount} through payment gateway...")


class RefundProcessor:
    def process(self, ticket):
        if not ticket.is_refundable():
            raise ValueError("Ticket is non-refundable")
        print(f"Refund processed for ${ticket.get_cost()}")


class TicketBookingSystem:
    def __init__(self):
        self.tickets = []
        self.payment = PaymentProcessor()
        self.refund = RefundProcessor()

    def book_ticket(self, ticket):
        self.tickets.append(ticket)
        print(f"Booked ticket for ${ticket.get_cost()}")
        self.payment.charge(ticket.get_cost())

    def process_refund(self, index):
        self.refund.process(self.tickets[index])

system = TicketBookingSystem()
system.book_ticket(EconomyTicket(100))
system.book_ticket(BusinessTicket(300))
system.process_refund(1)

Booked ticket for $100
Charging $100 through payment gateway...
Booked ticket for $300
Charging $300 through payment gateway...
Refund processed for $300


3. **Система управления бронированием для отелей (2б)**

В системе управления бронированием для отелей обрабатываются все аспекты бронирования: проверка доступности номеров, расчет стоимости, создание бронирования, отправка уведомлений клиенту и генерация отчетов для менеджеров.

Исходный код:

In [ ]:
import requests

class BookingManager:
    def __init__(self):
        self.rooms = {"101": True, "102": False, "103": True}  # True - свободно, False - занято

    def check_availability(self, room_id):
        return self.rooms.get(room_id, False)

    def calculate_price(self, room_id, nights):
        base_price = 100
        if room_id == "102":
            base_price = 150  # Цена люкса
        return base_price * nights

    def create_booking(self, room_id, customer_name, nights):
        if not self.check_availability(room_id):
            raise ValueError("Room not available")
        price = self.calculate_price(room_id, nights)
        self.rooms[room_id] = False
        print(f"Booking created for {customer_name} in room {room_id} for {nights} nights. Total price: ${price}.")
        return {"room_id": room_id, "customer_name": customer_name, "nights": nights, "price": price}

    def send_notification(self, customer_name, email):
        print(f"Sending booking confirmation to {customer_name} at {email}...")
        response = requests.post("https://example.com/notify", json={"name": customer_name, "email": email})
        if response.status_code == 200:
            print("Notification sent.")
        else:
            print("Failed to send notification.")

    def generate_report(self):
        booked_rooms = [room for room, available in self.rooms.items() if not available]
        print("Booked rooms:", booked_rooms)
        return booked_rooms

manager = BookingManager()
manager.create_booking("101", "Alice", 3)
manager.send_notification("Alice", "alice@example.com")
manager.generate_report()

Booking created for Alice in room 101 for 3 nights. Total price: $300.
Sending booking confirmation to Alice at alice@example.com...
Failed to send notification.
Booked rooms: ['101', '102']


['101', '102']

Проблемы:

1. **Нарушение SRP**:
   - Класс `BookingManager` обрабатывает очень много обязанностей, который можно разделить

2. **Нарушение OCP**:
   - Для изменения правил расчёта цены или добавления новых типов номеров нужно изменять метод `calculate_price`

3. **Нарушение DIP**:
   - Класс `BookingManager` напрямую зависит от библиотеки requests для отправки уведомлений

Решение:

In [8]:
from abc import ABC, abstractmethod

class PricingStrategy(ABC):
    @abstractmethod
    def calculate_price(self, nights):
        pass


class StandardRoomPrice(PricingStrategy):
    def calculate_price(self, nights):
        return 100 * nights


class LuxuryRoomPrice(PricingStrategy):
    def calculate_price(self, nights):
        return 150 * nights


class RoomAvailability:
    def __init__(self):
        self.rooms = {"101": True, "102": False, "103": True}

    def is_available(self, room_id):
        return self.rooms.get(room_id, False)

    def book_room(self, room_id):
        self.rooms[room_id] = False


class NotificationService:
    def send(self, customer_name, email):
        print(f"Sending booking confirmation to {customer_name} at {email}...")
        success = self._send_http_notification(customer_name, email)
        print("Notification sent." if success else "Failed to send notification.")

    def _send_http_notification(self, name, email):
        return True


class ReportGenerator:
    def generate(self, rooms):
        booked = [room for room, available in rooms.items() if not available]
        print("Booked rooms:", booked)
        return booked


class BookingManager:
    def __init__(self):
        self.availability = RoomAvailability()
        self.pricing = {"101": StandardRoomPrice(), "102": LuxuryRoomPrice(), "103": StandardRoomPrice()}
        self.notifier = NotificationService()
        self.reporter = ReportGenerator()

    def create_booking(self, room_id, customer_name, nights):
        if not self.availability.is_available(room_id):
            raise ValueError("Room not available")

        price = self.pricing[room_id].calculate_price(nights)
        self.availability.book_room(room_id)
        print(f"Booking for {customer_name} in room {room_id} for {nights} nights. Price: ${price}")

        self.notifier.send(customer_name, f"{customer_name}@example.com")
        return {"room_id": room_id, "customer_name": customer_name, "nights": nights, "price": price}

    def generate_report(self):
        return self.reporter.generate(self.availability.rooms)

manager = BookingManager()
manager.create_booking("101", "Alice", 3)
manager.generate_report()

Booking for Alice in room 101 for 3 nights. Price: $300
Sending booking confirmation to Alice at Alice@example.com...
Notification sent.
Booked rooms: ['101', '102']


['101', '102']

4. **Система управления логистикой (1б)**

В системе управления логистикой есть разные типы заказов: стандартные, экспресс и крупногабаритные. Каждый тип заказа имеет свои уникальные требования:  
- **Стандартные заказы** рассчитывают маршрут на основе ближайших пунктов доставки.  
- **Экспресс-заказы** имеют фиксированные маршруты, но для них также требуется учитывать время доставки.  
- **Крупногабаритные заказы** требуют предварительного расчета вместимости транспорта.

Исходный код:

In [11]:
class Order:
    def __init__(self, order_id, destination, weight):
        self.order_id = order_id
        self.destination = destination
        self.weight = weight

    def calculate_route(self):
        # Универсальный метод расчета маршрута
        print(f"Calculating generic route for order {self.order_id} to {self.destination}")

    def calculate_delivery_time(self):
        return 2  # Время в днях

    def calculate_capacity(self):
        return self.weight

class StandardOrder(Order):
    def calculate_route(self):
        print(f"Calculating optimized route for standard order {self.order_id} to {self.destination}")


class ExpressOrder(Order):
    def calculate_delivery_time(self):
        print(f"Calculating delivery time for express order {self.order_id}")
        return 1  # Экспресс-доставка за 1 день


class BulkOrder(Order):
    def calculate_capacity(self):
        print(f"Calculating capacity for bulk order {self.order_id}")
        return self.weight * 2  # Специальный расчет для крупногабаритных заказов

class LogisticsManager:
    def __init__(self):
        self.orders = []

    def add_order(self, order):
        self.orders.append(order)

    def process_orders(self):
        for order in self.orders:
            order.calculate_route()
            delivery_time = order.calculate_delivery_time()
            capacity = order.calculate_capacity()
            print(f"Order {order.order_id}: delivery time is {delivery_time} days, capacity {capacity}")

manager = LogisticsManager()
manager.add_order(StandardOrder("001", "New York", 10))
manager.add_order(ExpressOrder("002", "Los Angeles", 5))
manager.add_order(BulkOrder("003", "Chicago", 100))
manager.process_orders()

Calculating optimized route for standard order 001 to New York
Order 001: delivery time is 2 days, capacity 10
Calculating generic route for order 002 to Los Angeles
Calculating delivery time for express order 002
Order 002: delivery time is 1 days, capacity 5
Calculating generic route for order 003 to Chicago
Calculating capacity for bulk order 003
Order 003: delivery time is 2 days, capacity 200


Проблемы:

1. **Нарушение SRP**:
   - Класс `Order` отвечает за расчёт маршрута, времени доставки и вместимости одновременно

2. **Нарушение OCP**:
   - Чтобы добавить новый тип заказа с новой логикой, нужно изменять класс `Order` или создавать новые методы в наследниках, нарушая контракт

3. **Нарушение LSP**:
   - Дочерние классы нарушают ожидаемое поведение: `ExpressOrder.calculate_delivery_time()` печатает сообщение и возвращает значение, а родительский просто возвращает число

Решение:

In [12]:
class Order:
    def __init__(self, order_id, destination, weight):
        self.order_id = order_id
        self.destination = destination
        self.weight = weight

    def calculate_route(self):
        pass

    def calculate_delivery_time(self):
        return 2

    def calculate_capacity(self):
        return self.weight

    def process(self):
        raise NotImplementedError("Subclasses must implement process()")


class StandardOrder(Order):
    def calculate_route(self):
        print(f"Calculating optimized route for standard order {self.order_id} to {self.destination}")

    def process(self):
        self.calculate_route()
        delivery_time = self.calculate_delivery_time()
        capacity = self.calculate_capacity()
        print(f"Order {self.order_id}: delivery time is {delivery_time} days, capacity {capacity}")


class ExpressOrder(Order):
    def calculate_route(self):
        print(f"Calculating fixed route for express order {self.order_id} to {self.destination}")

    def calculate_delivery_time(self):
        return 1

    def process(self):
        self.calculate_route()
        print(f"Calculating delivery time for express order {self.order_id}")
        delivery_time = self.calculate_delivery_time()
        capacity = self.calculate_capacity()
        print(f"Order {self.order_id}: delivery time is {delivery_time} days, capacity {capacity}")


class BulkOrder(Order):
    def calculate_route(self):
        print(f"Calculating special route for bulk order {self.order_id} to {self.destination}")

    def calculate_capacity(self):
        return self.weight * 2

    def process(self):
        self.calculate_route()
        delivery_time = self.calculate_delivery_time()
        print(f"Calculating capacity for bulk order {self.order_id}")
        capacity = self.calculate_capacity()
        print(f"Order {self.order_id}: delivery time is {delivery_time} days, capacity {capacity}")


class LogisticsManager:
    def __init__(self):
        self.orders = []

    def add_order(self, order):
        self.orders.append(order)

    def process_orders(self):
        for order in self.orders:
            order.process()

manager = LogisticsManager()
manager.add_order(StandardOrder("001", "New York", 10))
manager.add_order(ExpressOrder("002", "Los Angeles", 5))
manager.add_order(BulkOrder("003", "Chicago", 100))
manager.process_orders()

Calculating optimized route for standard order 001 to New York
Order 001: delivery time is 2 days, capacity 10
Calculating fixed route for express order 002 to Los Angeles
Calculating delivery time for express order 002
Order 002: delivery time is 1 days, capacity 5
Calculating special route for bulk order 003 to Chicago
Calculating capacity for bulk order 003
Order 003: delivery time is 2 days, capacity 200


5. **Система обработки задач (5б)**

*Выполнять локально (не в colab).*

Разрабатываем систему обработки задач, использующую два типа очередей:
1. **Локальная очередь** — хранит задачи в оперативной памяти.
2. **Распределенная очередь** — взаимодействует с **RabbitMQ**.

Система выполняет три типа задач:
- `send_email`: Отправка email.
- `generate_report`: Генерация отчета.
- `process_data`: Обработка данных.


Для работы с **RabbitMQ** потребуется:
- Установленный и запущенный сервер RabbitMQ.
- Брокер сообщений будет доступен на порту `5672` для взаимодействия через AMQP-протокол.
- Веб-интерфейс управления RabbitMQ будет доступен по адресу [http://localhost:15672](http://localhost:15672), если вы используете официальный Docker-образ с поддержкой веб-интерфейса.

Чтобы запустить RabbitMQ, вы можете воспользоваться Docker. Это самый простой способ:

```bash
docker run -d --hostname my-rabbit --name some-rabbit \
-p 5672:5672 -p 15672:15672 rabbitmq:management
```

Если образ уже обновился, выполните команду для загрузки последней версии:

```bash
docker pull rabbitmq:management
```

После запуска:
- **Порт 5672** будет доступен для AMQP-протокола.
- **Порт 15672** будет доступен для веб-интерфейса RabbitMQ Management Console.
- Логин и пароль по умолчанию: `guest` / `guest`.

Убедитесь, что контейнер с RabbitMQ работает, выполнив команду.

```bash
docker ps
```

Перейдите в браузере на [http://localhost:15672](http://localhost:15672) для входа в веб-интерфейс с учетными данными `guest/guest`..

Исходный код:

In [ ]:
! pip install pika


[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
class TaskProcessor:
    def process_task(self, task):
        task_type = task.get("type")

        payload = task.get("payload")

        if task_type == "send_email":
            self._send_email(payload)
        elif task_type == "generate_report":
            self._generate_report(payload)
        elif task_type == "process_data":
            self._process_data(payload)
        else:
            print(f"Unknown task type: {task_type}")

    def _send_email(self, payload):
        email = payload.get("email")
        subject = payload.get("subject")
        content = payload.get("content")
        print(f"Sending email to {email} with subject '{subject}' and content '{content}'")

    def _generate_report(self, payload):
        report_id = payload.get("report_id")
        print(f"Generating report with ID {report_id}...")
        # Имитация долгой работы
        import time
        time.sleep(2)
        print(f"Report {report_id} generated")

    def _process_data(self, payload):
        data = payload.get("data")
        print(f"Processing data: {data}")
        # Имитация обработки данных
        processed_data = [d.upper() for d in data]
        print(f"Processed data: {processed_data}")

In [ ]:
import time
import json
import pika


class TaskQueue:
    def enqueue(self, task):
        pass

    def dequeue(self):
        pass

    def connect(self):
        pass


class LocalQueue(TaskQueue):
    def __init__(self):
        self.tasks = []

    def enqueue(self, task):
        self.tasks.append(task)

    def dequeue(self):
        if not self.tasks:
            return None
        return self.tasks.pop(0)


class RabbitMQQueue(TaskQueue):
    def __init__(self, broker_url, queue_name):
        self.broker_url = broker_url
        self.queue_name = queue_name
        self.channel = None

    def connect(self):
        connection = pika.BlockingConnection(pika.ConnectionParameters(self.broker_url))
        self.channel = connection.channel()
        self.channel.queue_declare(queue=self.queue_name, durable=True)
        print(f"Connected to RabbitMQ queue '{self.queue_name}'")

    def enqueue(self, task):
        """Добавляет задачу в RabbitMQ с сериализацией в JSON."""
        if not self.channel:
            raise ConnectionError("Not connected to RabbitMQ")
        task_str = json.dumps(task)  # Сериализуем задачу в строку
        self.channel.basic_publish(
            exchange="",
            routing_key=self.queue_name,
            body=task_str,
            properties=pika.BasicProperties(
                delivery_mode=2  # Делает сообщение устойчивым
            )
        )
        print(f"Task '{task_str}' enqueued in RabbitMQQueue")

    def dequeue(self):
        """Извлекает задачу из RabbitMQ с десериализацией из JSON."""
        if not self.channel:
            raise ConnectionError("Not connected to RabbitMQ")
        method_frame, header_frame, body = self.channel.basic_get(queue=self.queue_name)
        if method_frame:
            self.channel.basic_ack(method_frame.delivery_tag)
            task = json.loads(body.decode())  # Десериализуем задачу обратно в словарь
            print(f"Task '{task}' dequeued from RabbitMQQueue")
            return task
        return None


class TaskManager:
    def __init__(self, queue, processor):
        self.queue = queue
        self.processor = processor

    def process_tasks(self):
        while True:
            task = self.queue.dequeue()
            if task is None:
                print("No tasks to process. Waiting...")
                import time
                time.sleep(2)
            else:
                print(f"Processing task: {task}")
                self.processor.process_task(task)

In [ ]:
local_queue = LocalQueue()
processor = TaskProcessor()
manager = TaskManager(local_queue, processor)

# Добавляем задачи в локальную очередь
local_queue.enqueue({"type": "send_email", "payload": {"email": "user@example.com", "subject": "Welcome", "content": "Thank you for joining!"}})
local_queue.enqueue({"type": "generate_report", "payload": {"report_id": 42}})
local_queue.enqueue({"type": "process_data", "payload": {"data": ["foo", "bar", "baz"]}})

# Запускаем обработку задач
manager.process_tasks()

Processing task: {'type': 'send_email', 'payload': {'email': 'user@example.com', 'subject': 'Welcome', 'content': 'Thank you for joining!'}}
Sending email to user@example.com with subject 'Welcome' and content 'Thank you for joining!'
Processing task: {'type': 'generate_report', 'payload': {'report_id': 42}}
Generating report with ID 42...
Report 42 generated
Processing task: {'type': 'process_data', 'payload': {'data': ['foo', 'bar', 'baz']}}
Processing data: ['foo', 'bar', 'baz']
Processed data: ['FOO', 'BAR', 'BAZ']
No tasks to process. Waiting...
No tasks to process. Waiting...
No tasks to process. Waiting...


KeyboardInterrupt: 

In [ ]:
# Создаем очередь RabbitMQ
rabbit_queue = RabbitMQQueue("localhost", "task_queue")
rabbit_queue.connect()

# Создаем обработчик задач и менеджер
processor = TaskProcessor()
manager = TaskManager(rabbit_queue, processor)

# Добавляем задачи в очередь
rabbit_queue.enqueue({"type": "send_email", "payload": {"email": "admin@example.com", "subject": "Alert", "content": "Server is down!"}})
rabbit_queue.enqueue({"type": "generate_report", "payload": {"report_id": 101}})
rabbit_queue.enqueue({"type": "process_data", "payload": {"data": ["apple", "banana", "cherry"]}})

# Запускаем обработку задач
manager.process_tasks()

Connected to RabbitMQ queue 'task_queue'
Task '{"type": "send_email", "payload": {"email": "admin@example.com", "subject": "Alert", "content": "Server is down!"}}' enqueued in RabbitMQQueue
Task '{"type": "generate_report", "payload": {"report_id": 101}}' enqueued in RabbitMQQueue
Task '{"type": "process_data", "payload": {"data": ["apple", "banana", "cherry"]}}' enqueued in RabbitMQQueue
Task '{'type': 'send_email', 'payload': {'email': 'admin@example.com', 'subject': 'Alert', 'content': 'Server is down!'}}' dequeued from RabbitMQQueue
Processing task: {'type': 'send_email', 'payload': {'email': 'admin@example.com', 'subject': 'Alert', 'content': 'Server is down!'}}
Sending email to admin@example.com with subject 'Alert' and content 'Server is down!'
Task '{'type': 'generate_report', 'payload': {'report_id': 101}}' dequeued from RabbitMQQueue
Processing task: {'type': 'generate_report', 'payload': {'report_id': 101}}
Generating report with ID 101...
Report 101 generated
Task '{'type':

KeyboardInterrupt: 

Проверьте в веб-интерфейсе, что у вас появилась очередь сообщений `task_queue` и в ней было какое-то сообщение.

После нескольких секунд для каждого обработчика можно его прерывать, поскольку каждый из них будет просто ожидать данных и писать `No tasks to process. Waiting...`

Для удаления Docker-образа после использования вам нужно выполнить несколько шагов:

#### 1. Остановить и удалить контейнер

Если контейнер с RabbitMQ всё ещё работает, его нужно сначала остановить и удалить:

```bash
docker ps
```

Эта команда покажет список всех работающих контейнеров. Найдите контейнер RabbitMQ, например:

```
CONTAINER ID   IMAGE                COMMAND                  STATUS        NAMES
e0db3c8c0d8f   rabbitmq:management  "docker-entrypoint.s…"   Up 3 minutes  some-rabbit
```

Чтобы остановить контейнер, выполните:

```bash
docker stop some-rabbit
```

Теперь удалите остановленный контейнер:

```bash
docker rm some-rabbit
```

#### 2. Удалить Docker-образ

Сначала проверьте, какие образы загружены в вашей системе:

```bash
docker images
```

Вы увидите список образов:

```
REPOSITORY     TAG            IMAGE ID       CREATED        SIZE
rabbitmq       management     f41c9db5c2c5   2 weeks ago    188MB
```

Удалите образ RabbitMQ:

```bash
docker rmi rabbitmq:management
```
